# Notebook 15b — MJO SSL Temporal 2D Encoder (lat-aware, 16 × 180)
**Project:** ENSO-BSISO SSL — MJO Extension  
**Author:** Jiayi (jh9141@nyu.edu)

Variant of `nb15` using the **lat-aware preprocessing** from `nb13b`. Input shape is `(N, 3, 16, 180)` — the 15°S–15°N latitude strip is preserved through the bandpass and through the encoder. Self-supervised contrastive learning with temporal-proximity pairs, no RMM/ENSO labels.

Why: Session 24's `nb15` produced a severe month-clustering confound (angle ANOVA F = 300.84). The hypothesis is that the meridional average removed the N–S structure the encoder needed to distinguish MJO state from seasonal background. Preserving the lat axis should let the encoder learn off-equator Rossby gyres, ITCZ asymmetry, and monsoon signal — features that change with MJO phase independently of calendar month.

## Setup (locked decisions — Session 25)

| Knob | Value | Source |
|---|---|---|
| Input | `X_MJO_lat16.npy`, shape `(N, 3, 16, 180)` | nb13b |
| Embedding dim | 2 (no L2 norm) | nb07c/nb08 lesson |
| Loss | raw dot product InfoNCE | nb08 |
| Temperature τ | 0.5 | nb08 |
| Weight decay | 1e-4 | nb08 |
| Optimizer | Adam, lr=1e-3, cosine→1e-5 | nb08 |
| Epochs | 50 | nb08 |
| Architecture | 4 lat-pool blocks (k=3×3) + 2 lon-pool blocks (k=1×3) | Session 25 Q2 |
| **Bandpass** | **Lanczos 20–90 d, applied per `(lat, lon)`** | Session 25 |
| Edge drop | 90 days from each end of full record | filter edge effects |
| Positive pair | anchor d, positive in [d−3, d+3] \ {d}, same year | nb08 |
| Year split | every 5th year held out | nb04/nb07c/nb08 |

## Outputs

- `MJO/lat16/checkpoints/encoder_mjo_ssl_lat16_final.pth`
- `MJO/lat16/data/processed/X_MJO_lat16_bp20_90.npy` — bandpassed input over full (lat, lon) grid (saved for reuse by nb16b)
- `MJO/lat16/data/processed/labels_aligned_mjo_lat16_bp20_90.csv` — labels after edge drops
- `MJO/lat16/results/ssl/embeddings.npy`, `training_curves.png`, `embedding_2d_overview.png`, `month_clustering_check.png`, `linear_probe_results.json`, `enso_displacement.png`, `mjo_ssl_lat16_summary.md`

## Runtime
Bandpass over full (lat, lon) grid: ~3–10 min (~16× more data than nb15 but still trivial for `scipy.ndimage.convolve1d`). Training: ~30–45 min on T4. Total ~35–55 min.

**Memory note.** Peak during bandpass ≈ 3 GB (raw X + intermediate float64 + filtered output). Well within T4's 16 GB.

---

## Cell 1 — Mount Drive + Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, json, time, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from scipy.ndimage import convolve1d

EMBEDDING_DIM   = 2
RUN_TAG         = 'mjo_ssl_lat16'
TEMPERATURE     = 0.5
EPOCHS          = 50
BATCH_SIZE      = 64
LR              = 1e-3
WEIGHT_DECAY    = 1e-4
MAX_DELTA       = 3        # positive pair window: [d-3, d+3] \ {d}, same year
BP_LOW_DAYS     = 20
BP_HIGH_DAYS    = 90
BP_HALF_W       = 90       # filter half-window in days (filter length = 181)
EDGE_DROP_DAYS  = BP_HALF_W

PROJECT_DIR    = '/content/drive/MyDrive/BSISO_SSL_Project'
MJO_DIR        = f'{PROJECT_DIR}/MJO'
LAT16_DIR      = f'{MJO_DIR}/lat16'
PROCESSED_DIR  = f'{LAT16_DIR}/data/processed'
CHECKPOINT_DIR = f'{LAT16_DIR}/checkpoints'
RESULTS_DIR    = f'{LAT16_DIR}/results/ssl'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

X_FILE         = 'X_MJO_lat16.npy'
LABELS_FILE    = 'labels_aligned_mjo_lat16.csv'
X_BP_FILE      = 'X_MJO_lat16_bp20_90.npy'
LABELS_BP_FILE = 'labels_aligned_mjo_lat16_bp20_90.csv'

# Clear stale outputs from any prior nb15b runs
stale = (glob.glob(f'{CHECKPOINT_DIR}/encoder_{RUN_TAG}_*.pth') +
         glob.glob(f'{CHECKPOINT_DIR}/training_history_{RUN_TAG}.json') +
         glob.glob(f'{RESULTS_DIR}/*'))
for p in stale:
    try: os.remove(p)
    except (IsADirectoryError, FileNotFoundError): pass
print(f'Cleared {len(stale)} stale files from previous runs.')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device:           {device}')
print(f'Embedding dim:    {EMBEDDING_DIM} (no L2 normalization)')
print(f'Loss:             raw dot product InfoNCE (τ={TEMPERATURE})')
print(f'Bandpass:         Lanczos {BP_LOW_DAYS}–{BP_HIGH_DAYS} d  (half_w={BP_HALF_W}, edge drop ±{EDGE_DROP_DAYS} d, per (lat, lon))')
print(f'Positive pair:    [d−{MAX_DELTA}, d+{MAX_DELTA}] \\ {{d}}, same year')
print(f'Run tag:          {RUN_TAG}')

## Cell 2 — Load Data

In [ ]:
X      = np.load(f'{PROCESSED_DIR}/{X_FILE}')
labels = pd.read_csv(f'{PROCESSED_DIR}/{LABELS_FILE}', parse_dates=['date'])

print(f'X shape:  {X.shape}  (expect (N, 3, 16, 180))')
print(f'Labels:   {len(labels)} rows')
assert X.shape[0] == len(labels), 'X / labels length mismatch'
assert X.shape[1:] == (3, 16, 180), f'Unexpected shape {X.shape[1:]}'

print(f'\nDate range:    {labels["date"].min().date()} to {labels["date"].max().date()}')
print(f'Month coverage: {sorted(labels["date"].dt.month.unique())}')
print(f'Memory:         {X.nbytes / 1e6:.0f} MB (float32)')

## Cell 3 — Lanczos 20–90 d Bandpass over `(lat, lon)`

**Filter design** (identical to nb15):
- Bandpass = lowpass(20 d) − lowpass(90 d) → passes periods in (20, 90) d.
- Half-window 90 d → filter length 181 taps.
- Applied to the **full continuous time series** along the time axis. `scipy.ndimage.convolve1d(X, weights, axis=0)` broadcasts across the spatial dims `(channels, lat, lon)`, so each `(channel, lat, lon)` grid point is filtered independently.
- Drop first/last 90 days only.

**Difference from nb15.** Same filter; same `axis=0` convolve. The only change is that the spatial dims are `(3, 16, 180)` instead of `(3, 1, 180)`, so ~16× more grid points get filtered. Still fast (~minutes).

In [ ]:
def lanczos_lowpass_weights(cutoff_days, half_window):
    """Lanczos-windowed sinc lowpass. Passes periods > cutoff_days."""
    k = np.arange(-half_window, half_window + 1)
    fc = 1.0 / cutoff_days
    h = np.where(k == 0, 2*fc, np.sin(2*np.pi*fc*k) / (np.pi*k))
    sigma = np.where(k == 0, 1.0, np.sin(np.pi*k/half_window) / (np.pi*k/half_window))
    w = h * sigma
    w /= w.sum()
    return w

def lanczos_bandpass_weights(low_period, high_period, half_window):
    """Bandpass passing periods in (low_period, high_period). low < high in days."""
    w_low  = lanczos_lowpass_weights(low_period,  half_window)
    w_high = lanczos_lowpass_weights(high_period, half_window)
    return w_low - w_high

weights = lanczos_bandpass_weights(BP_LOW_DAYS, BP_HIGH_DAYS, BP_HALF_W)
print(f'Filter length:    {len(weights)} taps (half_w={BP_HALF_W})')
print(f'Sum of weights:   {weights.sum():.6f}  (should be ≈ 0 — bandpass has zero DC gain)')
print(f'Peak |weight|:    {np.abs(weights).max():.4f}')

# Frequency response (sanity)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].stem(np.arange(-BP_HALF_W, BP_HALF_W+1), weights, basefmt=' ')
axes[0].set_xlabel('k (days)'); axes[0].set_ylabel('weight')
axes[0].set_title(f'Lanczos {BP_LOW_DAYS}–{BP_HIGH_DAYS} d bandpass impulse', fontweight='bold')
axes[0].grid(alpha=0.3)

freqs = np.linspace(0.001, 0.5, 600)
k_arr = np.arange(-BP_HALF_W, BP_HALF_W+1)
response = np.abs(np.array([np.sum(weights * np.exp(-2j*np.pi*f*k_arr)) for f in freqs]))
periods = 1.0 / freqs
axes[1].plot(periods, response, linewidth=2)
axes[1].axvline(BP_LOW_DAYS,  color='red',  linestyle='--', label=f'low cutoff {BP_LOW_DAYS} d')
axes[1].axvline(BP_HIGH_DAYS, color='blue', linestyle='--', label=f'high cutoff {BP_HIGH_DAYS} d')
axes[1].set_xscale('log'); axes[1].set_xlabel('period (days)'); axes[1].set_ylabel('|H(f)|')
axes[1].set_title('Frequency response (passband ≈ 20–90 d)', fontweight='bold')
axes[1].set_xlim(2, 500); axes[1].legend(); axes[1].grid(alpha=0.3, which='both')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/bandpass_filter_response.png', dpi=130, bbox_inches='tight')
plt.show()

# Apply filter to full continuous record over the entire (lat, lon) grid
print(f'\nApplying bandpass to full continuous time series — shape {X.shape} ...')
t0 = time.time()
X_filt = convolve1d(X.astype(np.float64), weights, axis=0, mode='reflect').astype(np.float32)
print(f'Bandpass complete in {time.time()-t0:.1f} s')

# Drop edges
T = len(X)
mask_keep = np.zeros(T, dtype=bool)
mask_keep[EDGE_DROP_DAYS : T - EDGE_DROP_DAYS] = True

X_bp = X_filt[mask_keep]
labels_bp = labels.loc[mask_keep].reset_index(drop=True)

print(f'\nBefore drop: {T} samples')
print(f'After drop:  {len(X_bp)} samples  ({100*len(X_bp)/T:.1f}%)')
print(f'Date range:  {labels_bp["date"].min().date()} to {labels_bp["date"].max().date()}')

np.save(f'{PROCESSED_DIR}/{X_BP_FILE}', X_bp)
labels_bp.to_csv(f'{PROCESSED_DIR}/{LABELS_BP_FILE}', index=False)
print(f'\nSaved: {X_BP_FILE} shape {X_bp.shape}  ({X_bp.nbytes/1e6:.0f} MB)')
print(f'Saved: {LABELS_BP_FILE}  rows {len(labels_bp)}')

# Variance comparison (per channel; spatial dims collapsed into one)
print('\nVariance (per channel, over all space-time samples kept) — raw vs bandpassed:')
for ch, name in enumerate(['u850', 'OLR', 'u200']):
    v_raw = X[mask_keep, ch].var()
    v_bp  = X_bp[:, ch].var()
    print(f'  {name}: raw var={v_raw:.4f}, bp var={v_bp:.4f}  (ratio={v_bp/v_raw:.2%})')

del X_filt
gc.collect()

## Cell 4 — Year-Based Train/Val Split (on Bandpassed Data)

In [ ]:
all_years = sorted(labels_bp['date'].dt.year.unique())
val_years = all_years[::5]
train_years = [y for y in all_years if y not in val_years]

year_col = labels_bp['date'].dt.year
train_idx = labels_bp.index[year_col.isin(train_years)].values
val_idx   = labels_bp.index[year_col.isin(val_years)].values

print(f'Val years ({len(val_years)}): {val_years}')
print(f'Train: {len(train_idx)} samples ({100*len(train_idx)/len(labels_bp):.1f}%)')
print(f'Val:   {len(val_idx)} samples ({100*len(val_idx)/len(labels_bp):.1f}%)')
print(f'\nENSO distribution (val):')
print(labels_bp.loc[val_idx, 'enso_category'].value_counts())

## Cell 5 — SSL Temporal Pair Sampler + Dataset

Same logic as nb15 / nb08. Pairs defined by temporal proximity only — model never sees RMM phase or ENSO during training.

In [ ]:
class TemporalPairSampler:
    def __init__(self, labels_df, allowed_indices, max_delta=3):
        self.labels = labels_df
        self.allowed_set = set(int(i) for i in allowed_indices)
        self.max_delta = max_delta
        sub = labels_df.loc[list(self.allowed_set)]
        self.date_to_idx = {pd.Timestamp(d): int(i)
                            for d, i in zip(sub['date'].values, sub.index.values)}

    def sample_positive_pair(self, anchor_idx):
        anchor_date = self.labels.loc[anchor_idx, 'date']
        anchor_year = anchor_date.year
        candidates = []
        for delta in range(-self.max_delta, self.max_delta + 1):
            if delta == 0: continue
            target = anchor_date + pd.Timedelta(days=delta)
            if target.year != anchor_year: continue
            key = pd.Timestamp(target)
            if key in self.date_to_idx:
                candidates.append(self.date_to_idx[key])
        if not candidates:
            return anchor_idx, anchor_idx
        return anchor_idx, int(np.random.choice(candidates))


class SSLTemporalDataset(Dataset):
    def __init__(self, X_data, labels_df, train_indices, max_delta=3, mode='train', n_val_pairs=1000):
        self.X = X_data
        self.labels = labels_df
        self.train_indices = np.asarray(train_indices)
        self.sampler = TemporalPairSampler(labels_df, train_indices, max_delta)
        self.mode = mode
        if mode == 'val':
            rng = np.random.default_rng(42)
            self.val_pairs = []
            for idx in rng.choice(self.train_indices,
                                  size=min(n_val_pairs * 2, len(self.train_indices)), replace=False):
                a, b = self.sampler.sample_positive_pair(int(idx))
                if a != b: self.val_pairs.append((a, b))
                if len(self.val_pairs) >= n_val_pairs: break

    def __len__(self):
        return len(self.train_indices) if self.mode == 'train' else len(self.val_pairs)

    def __getitem__(self, idx):
        if self.mode == 'train':
            anchor = int(self.train_indices[idx])
            a, b = self.sampler.sample_positive_pair(anchor)
        else:
            a, b = self.val_pairs[idx]
        return torch.from_numpy(self.X[a]).float(), torch.from_numpy(self.X[b]).float()


train_dataset = SSLTemporalDataset(X_bp, labels_bp, train_idx, max_delta=MAX_DELTA, mode='train')
val_dataset   = SSLTemporalDataset(X_bp, labels_bp, val_idx,   max_delta=MAX_DELTA, mode='val')

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_dataset)} items → {len(train_loader)} batches/epoch')
print(f'Val:   {len(val_dataset)} pairs  → {len(val_loader)} batches')

# Sampler sanity
n_zero = 0
print('\nSample positive pairs (anchor → positive):')
for i in range(10):
    a, b = train_dataset.sampler.sample_positive_pair(int(train_idx[i]))
    da = labels_bp.loc[a, 'date']; db = labels_bp.loc[b, 'date']
    delta_d = (db - da).days
    if delta_d == 0: n_zero += 1
    print(f'  {da.date()} (P{labels_bp.loc[a, "phase"]}) → {db.date()} (P{labels_bp.loc[b, "phase"]})  Δ={delta_d}d')
assert n_zero == 0, f'{n_zero}/10 pairs are (anchor, anchor) — sampler broken.'
assert len(val_dataset) > 0, 'val_dataset is empty.'
print('\n✓ Sampler healthy.')

## Cell 6 — Lat-Aware CNN Encoder + Loss (same architecture as nb14b)

Same `MJOEncoderNoL2Lat16` as nb14b — apples-to-apples comparison. 4 lat-pool blocks (3×3 conv + lat-only MaxPool collapsing 16→1) followed by 2 lon-pool blocks identical to nb15's design. ~30 K params.

In [ ]:
class MJOEncoderNoL2Lat16(nn.Module):
    """Lat-aware encoder for (B, 3, 16, 180). Progressive lat-only pool
    collapses lat 16→8→4→2→1; then lon-only convs match nb15."""
    def __init__(self, embedding_dim=2):
        super().__init__()
        # Lat-compression: 3x3 convs with lat-only MaxPool
        self.conv1 = nn.Conv2d(3,  16, kernel_size=(3, 3), padding=(1, 1), bias=False)
        self.bn1   = nn.BatchNorm2d(16)
        self.pool1 = nn.MaxPool2d((2, 1))
        self.conv2 = nn.Conv2d(16, 32, kernel_size=(3, 3), padding=(1, 1), bias=False)
        self.bn2   = nn.BatchNorm2d(32)
        self.pool2 = nn.MaxPool2d((2, 1))
        self.conv3 = nn.Conv2d(32, 32, kernel_size=(3, 3), padding=(1, 1), bias=False)
        self.bn3   = nn.BatchNorm2d(32)
        self.pool3 = nn.MaxPool2d((2, 1))
        self.conv4 = nn.Conv2d(32, 32, kernel_size=(3, 3), padding=(1, 1), bias=False)
        self.bn4   = nn.BatchNorm2d(32)
        self.pool4 = nn.MaxPool2d((2, 1))
        # Lon-compression: 1x3 convs with lon-only MaxPool (matches nb15)
        self.conv5 = nn.Conv2d(32, 32, kernel_size=(1, 3), padding=(0, 1), bias=False)
        self.bn5   = nn.BatchNorm2d(32)
        self.pool5 = nn.MaxPool2d((1, 2))
        self.conv6 = nn.Conv2d(32, 32, kernel_size=(1, 3), padding=(0, 1), bias=False)
        self.bn6   = nn.BatchNorm2d(32)
        self.pool6 = nn.MaxPool2d((1, 2))
        # Head
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc          = nn.Linear(32, embedding_dim)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01); nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.pool3(F.relu(self.bn3(self.conv3(x))))
        x = self.pool4(F.relu(self.bn4(self.conv4(x))))
        x = self.pool5(F.relu(self.bn5(self.conv5(x))))
        x = self.pool6(F.relu(self.bn6(self.conv6(x))))
        x = self.global_pool(x).view(x.size(0), -1)
        return self.fc(x)


def InfoNCE_loss_raw(z_A, z_B, temperature):
    sim_matrix = torch.matmul(z_A, z_B.T) / temperature
    labels_ = torch.arange(z_A.size(0), device=z_A.device)
    return F.cross_entropy(sim_matrix, labels_)

encoder = MJOEncoderNoL2Lat16(embedding_dim=EMBEDDING_DIM).to(device)
optimizer = optim.Adam(encoder.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

# Sanity: forward pass on a lat16 batch
with torch.no_grad():
    dummy = torch.randn(4, 3, 16, 180).to(device)
    out = encoder(dummy)
    print(f'Parameters: {sum(p.numel() for p in encoder.parameters()):,}')
    print(f'Dummy forward: input {tuple(dummy.shape)} → output {tuple(out.shape)} (expect (4, {EMBEDDING_DIM}))')
print(f'Training:   {EPOCHS} epochs, batch {BATCH_SIZE}, τ={TEMPERATURE}, weight_decay={WEIGHT_DECAY}')

## Cell 7 — Training Loop

In [ ]:
from tqdm.notebook import tqdm
assert len(train_loader) > 0 and len(val_loader) > 0

history = {'train_loss': [], 'val_loss': [], 'epoch_time': [],
           'mean_norm': [], 'std_norm': [], 'max_norm': []}

for epoch in range(EPOCHS):
    t0 = time.time()
    encoder.train()
    train_loss = 0.0
    epoch_norms = []
    pbar = tqdm(train_loader, desc=f'ep {epoch+1}/{EPOCHS}', leave=False)
    for fA, fB in pbar:
        fA = fA.to(device, non_blocking=True); fB = fB.to(device, non_blocking=True)
        zA = encoder(fA); zB = encoder(fB)
        with torch.no_grad():
            epoch_norms.extend(zA.norm(dim=1).cpu().numpy().tolist())
            epoch_norms.extend(zB.norm(dim=1).cpu().numpy().tolist())
        loss = InfoNCE_loss_raw(zA, zB, temperature=TEMPERATURE)
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(encoder.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    train_loss /= len(train_loader)

    encoder.eval()
    val_loss = 0.0
    with torch.no_grad():
        for fA, fB in val_loader:
            fA = fA.to(device, non_blocking=True); fB = fB.to(device, non_blocking=True)
            zA = encoder(fA); zB = encoder(fB)
            val_loss += InfoNCE_loss_raw(zA, zB, temperature=TEMPERATURE).item()
    val_loss = val_loss / len(val_loader)
    scheduler.step()

    en = np.array(epoch_norms); et = time.time() - t0
    history['train_loss'].append(train_loss); history['val_loss'].append(val_loss)
    history['epoch_time'].append(et)
    history['mean_norm'].append(float(en.mean())); history['std_norm'].append(float(en.std()))
    history['max_norm'].append(float(en.max()))
    print(f'ep {epoch+1:2d}/{EPOCHS}  train={train_loss:.4f}  val={val_loss:.4f}  '
          f'norm μ={en.mean():.3f} σ={en.std():.3f} max={en.max():.2f}  '
          f'lr={scheduler.get_last_lr()[0]:.2e}  time={et:.1f}s')

    if (epoch + 1) % 10 == 0:
        torch.save({'epoch': epoch+1, 'model_state_dict': encoder.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'train_loss': train_loss, 'val_loss': val_loss},
                   f'{CHECKPOINT_DIR}/encoder_{RUN_TAG}_epoch_{epoch+1}.pth')

torch.save(encoder.state_dict(), f'{CHECKPOINT_DIR}/encoder_{RUN_TAG}_final.pth')
with open(f'{CHECKPOINT_DIR}/training_history_{RUN_TAG}.json', 'w') as f:
    json.dump(history, f, indent=2)

print(f'\nTraining complete. Total time: {sum(history["epoch_time"])/60:.1f} min')
print(f'Final mean norm: {history["mean_norm"][-1]:.3f}  max norm: {history["max_norm"][-1]:.2f}')

## Cell 8 — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
axes[0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0].plot(history['val_loss'],   label='Val',   linewidth=2)
axes[0].axhline(np.log(64), color='gray', linestyle='--', alpha=0.5, label='log(64)')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('InfoNCE Loss')
axes[0].set_title('SSL Training Curves (lat16)', fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history['mean_norm'], label='Mean norm', linewidth=2)
axes[1].fill_between(range(len(history['mean_norm'])),
                     np.array(history['mean_norm']) - np.array(history['std_norm']),
                     np.array(history['mean_norm']) + np.array(history['std_norm']),
                     alpha=0.3, label='±1σ')
axes[1].plot(history['max_norm'], label='Max norm', color='red', linestyle='--', linewidth=1)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Embedding norm')
axes[1].set_title('Norm Trajectory', fontweight='bold')
axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(history['epoch_time'], color='green', linewidth=2)
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Time (s)')
axes[2].set_title('Epoch Time', fontweight='bold')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
if max(history['max_norm']) > 100:
    print('\n⚠️  Norm explosion. Increase WEIGHT_DECAY.')
else:
    print(f'\n✓ Norms stable (max ever = {max(history["max_norm"]):.2f})')

## Cell 9 — Extract Embeddings

In [ ]:
encoder.eval()
embeddings_2d = np.zeros((len(X_bp), EMBEDDING_DIM), dtype=np.float32)
with torch.no_grad():
    for start in range(0, len(X_bp), 128):
        end = min(start + 128, len(X_bp))
        batch = torch.from_numpy(X_bp[start:end]).float().to(device)
        embeddings_2d[start:end] = encoder(batch).cpu().numpy()

np.save(f'{RESULTS_DIR}/embeddings.npy', embeddings_2d)
norms = np.linalg.norm(embeddings_2d, axis=1)
angles = np.arctan2(embeddings_2d[:, 1], embeddings_2d[:, 0])
print(f'Embeddings shape: {embeddings_2d.shape}')
print(f'Norm:  min={norms.min():.3f}  max={norms.max():.3f}  mean={norms.mean():.3f}  std={norms.std():.3f}')
print(f'Angle: spread = {angles.max() - angles.min():.3f} rad')

## Cell 10 — 4-Panel Scatter: by Phase, ENSO, Calendar Month, Amplitude

**Critical confound check (panel 3 — calendar month):** the central hypothesis of Session 25 is that the lat-aware encoder will see less of the seasonal cycle as discriminative information. If the by-month panel shows organized clusters (each month in its own region), seasonal contamination remains. If month colors are well-mixed, the encoder is tracking MJO state instead.

In [ ]:
lv = labels_bp.loc[val_idx]
act_v = (~lv['weak_mjo'].values) & (lv['phase'].between(1, 8).values)
Z_val = embeddings_2d[val_idx][act_v]
lv_a  = lv[act_v]

phase_colors = plt.cm.tab10(np.linspace(0, 0.8, 8))
enso_palette = {'El Nino': '#d62728', 'Neutral': '#7f7f7f', 'La Nina': '#1f77b4'}
enso_marker  = {'El Nino': '^',       'Neutral': 'o',        'La Nina': 's'}
month_colors = plt.cm.twilight(np.linspace(0, 1, 13))[:12]
month_names  = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

rng_lo = min(Z_val.min(), -1.0); rng_hi = max(Z_val.max(), 1.0)
pad = 0.1 * (rng_hi - rng_lo); rng_lo -= pad; rng_hi += pad

fig, axes = plt.subplots(1, 4, figsize=(24, 6))

# (a) by RMM phase
ax = axes[0]
for ph in range(1, 9):
    m = lv_a['phase'] == ph
    ax.scatter(Z_val[m, 0], Z_val[m, 1], c=[phase_colors[ph-1]], s=10, alpha=0.7, label=f'P{ph}')
ax.set_title('By RMM Phase\n(SSL never saw phase labels)', fontweight='bold')
ax.set_xlabel('z₁'); ax.set_ylabel('z₂')
ax.set_aspect('equal'); ax.set_xlim(rng_lo, rng_hi); ax.set_ylim(rng_lo, rng_hi)
ax.axhline(0, color='k', lw=0.4, alpha=0.3); ax.axvline(0, color='k', lw=0.4, alpha=0.3)
ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.2)

# (b) by ENSO
ax = axes[1]
for cat in ['El Nino', 'Neutral', 'La Nina']:
    m = lv_a['enso_category'] == cat
    ax.scatter(Z_val[m, 0], Z_val[m, 1], c=enso_palette[cat], marker=enso_marker[cat], s=10, alpha=0.5, label=cat)
ax.set_title('By ENSO\n(SSL never saw ENSO labels)', fontweight='bold')
ax.set_xlabel('z₁'); ax.set_ylabel('z₂')
ax.set_aspect('equal'); ax.set_xlim(rng_lo, rng_hi); ax.set_ylim(rng_lo, rng_hi)
ax.axhline(0, color='k', lw=0.4, alpha=0.3); ax.axvline(0, color='k', lw=0.4, alpha=0.3)
ax.legend(); ax.grid(alpha=0.2)

# (c) by CALENDAR MONTH — the critical confound check
ax = axes[2]
months = lv_a['date'].dt.month.values
for mi in range(1, 13):
    m = months == mi
    if m.sum() == 0: continue
    ax.scatter(Z_val[m, 0], Z_val[m, 1], c=[month_colors[mi-1]], s=10, alpha=0.6, label=month_names[mi-1])
ax.set_title('By Calendar Month\n(should NOT cluster by month)', fontweight='bold', color='red')
ax.set_xlabel('z₁'); ax.set_ylabel('z₂')
ax.set_aspect('equal'); ax.set_xlim(rng_lo, rng_hi); ax.set_ylim(rng_lo, rng_hi)
ax.axhline(0, color='k', lw=0.4, alpha=0.3); ax.axvline(0, color='k', lw=0.4, alpha=0.3)
ax.legend(fontsize=7, ncol=2, loc='best'); ax.grid(alpha=0.2)

# (d) by amplitude
ax = axes[3]
sc = ax.scatter(Z_val[:, 0], Z_val[:, 1], c=lv_a['amplitude'].values, cmap='viridis', s=10, alpha=0.7)
ax.set_title('By RMM Amplitude', fontweight='bold')
ax.set_xlabel('z₁'); ax.set_ylabel('z₂')
ax.set_aspect('equal'); ax.set_xlim(rng_lo, rng_hi); ax.set_ylim(rng_lo, rng_hi)
ax.axhline(0, color='k', lw=0.4, alpha=0.3); ax.axvline(0, color='k', lw=0.4, alpha=0.3)
plt.colorbar(sc, ax=ax, label='RMM amplitude'); ax.grid(alpha=0.2)

plt.suptitle(f'MJO SSL Temporal 2D (lat16) — Val Embeddings (active MJO, n={len(Z_val)})',
             fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/embedding_2d_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 11 — Month-Clustering Confound Check (formal ANOVA)

**This is the headline diagnostic of Session 25.** The nb15 SSL angle ANOVA F = 300.84 was the symptom that motivated the lat-aware redesign. Target for nb15b: angle ANOVA F < 50 on all-year data (matches BSISO MJJAS SSL acceptance). A drop from 300 → < 50 would be strong evidence that preserving the lat axis solves the seasonal contamination.

In [ ]:
from scipy.stats import f_oneway

angles_all = np.arctan2(embeddings_2d[:, 1], embeddings_2d[:, 0])
radii_all  = np.linalg.norm(embeddings_2d, axis=1)
months_all = labels_bp['date'].dt.month.values

angles_by_month = [angles_all[months_all == m] for m in range(1, 13)]
radii_by_month  = [radii_all[months_all == m]  for m in range(1, 13)]

f_a, p_a = f_oneway(*angles_by_month)
f_r, p_r = f_oneway(*radii_by_month)
print(f'Angle by month  ANOVA: F = {f_a:.2f}  p = {p_a:.2e}   (nb15: F = 300.84)')
print(f'Radius by month ANOVA: F = {f_r:.2f}  p = {p_r:.2e}   (nb15: F = 214.60)')

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
ax = axes[0]
bp = ax.boxplot(angles_by_month, positions=range(1, 13), widths=0.6, patch_artist=True, showfliers=False)
for patch, color in zip(bp['boxes'], plt.cm.twilight(np.linspace(0, 1, 13))[:12]):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
ax.set_xlabel('Calendar month'); ax.set_ylabel('Embedding angle θ (rad)')
ax.set_title(f'Angle by Month  ANOVA F = {f_a:.2f}  (p = {p_a:.2e})', fontweight='bold')
ax.grid(alpha=0.3)

ax = axes[1]
bp = ax.boxplot(radii_by_month, positions=range(1, 13), widths=0.6, patch_artist=True, showfliers=False)
for patch, color in zip(bp['boxes'], plt.cm.twilight(np.linspace(0, 1, 13))[:12]):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
ax.set_xlabel('Calendar month'); ax.set_ylabel('Embedding radius')
ax.set_title(f'Radius by Month  ANOVA F = {f_r:.2f}  (p = {p_r:.2e})', fontweight='bold')
ax.grid(alpha=0.3)

plt.suptitle('Month-Clustering Confound Check (MJO SSL lat16)', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/month_clustering_check.png', dpi=150, bbox_inches='tight')
plt.show()

if f_a > 50:
    print(f'\n⚠️  Strong month confound on angle (F={f_a:.0f}). Lat-aware redesign did NOT resolve the seasonal contamination. Consider tightening bandpass to (20, 60) d or restricting negatives to same calendar month.')
elif f_a > 10:
    print(f'\n△ Moderate month dependence (F={f_a:.1f}). Better than nb15 but not fully resolved. May reflect real intraseasonal amplitude modulation across the annual cycle.')
else:
    print(f'\n✓ Month dependence on angle is weak (F={f_a:.1f}). Lat-aware redesign resolved the seasonal contamination from nb15 (F=300.84).')

## Cell 12 — Linear Probes (RMM Phase + ENSO Balanced)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report
from sklearn.model_selection import cross_val_score, GroupKFold

active_mask = (~labels_bp['weak_mjo'].values) & (labels_bp['phase'].between(1, 8).values)
active_idx = np.where(active_mask)[0]
train_act  = np.intersect1d(train_idx, active_idx)
val_act    = np.intersect1d(val_idx,   active_idx)

Z_train = embeddings_2d[train_act]
Z_val_  = embeddings_2d[val_act]
year_groups_act = labels_bp.loc[active_idx, 'date'].dt.year.values
gkf = GroupKFold(n_splits=5)

# RMM phase
y_tr = labels_bp.loc[train_act, 'phase'].values
y_va = labels_bp.loc[val_act,   'phase'].values
clf = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
clf.fit(Z_train, y_tr)
phase_val = float(accuracy_score(y_va, clf.predict(Z_val_)))
cv_phase = cross_val_score(clf, embeddings_2d[active_idx], labels_bp.loc[active_idx, 'phase'].values,
                            cv=gkf, groups=year_groups_act, scoring='accuracy', n_jobs=-1)

# ENSO balanced
y_tr_e = labels_bp.loc[train_act, 'enso_category'].values
y_va_e = labels_bp.loc[val_act,   'enso_category'].values
clf_b = LogisticRegression(max_iter=1000, C=1.0, random_state=42, class_weight='balanced')
clf_b.fit(Z_train, y_tr_e)
enso_bal = float(balanced_accuracy_score(y_va_e, clf_b.predict(Z_val_)))
cv_enso = cross_val_score(clf_b, embeddings_2d[active_idx], labels_bp.loc[active_idx, 'enso_category'].values,
                           cv=gkf, groups=year_groups_act, scoring='balanced_accuracy', n_jobs=-1)

probe_results = {
    'RMM Phase': {'val_acc': phase_val,  'cv_mean': float(cv_phase.mean()), 'cv_std': float(cv_phase.std())},
    'ENSO bal':  {'val_acc': enso_bal,   'cv_mean': float(cv_enso.mean()),  'cv_std': float(cv_enso.std())},
}

print('=' * 70)
print('MJO SSL LAT16 — LINEAR PROBE RESULTS (no labels seen during training)')
print('=' * 70)
print(f'RMM phase val acc:    {phase_val*100:.1f}%   (random 12.5%;  nb15 was 24.7%)')
print(f'RMM phase 5-fold CV:  {cv_phase.mean()*100:.1f}% ± {cv_phase.std()*100:.1f}%')
print(f'ENSO  bal-acc val:    {enso_bal*100:.1f}%   (random 33.3%)')
print(f'ENSO  bal-acc 5-fold: {cv_enso.mean()*100:.1f}% ± {cv_enso.std()*100:.1f}%')
print(f'\nReference (BSISO SSL): phase ~30-40%, z=14.55')

with open(f'{RESULTS_DIR}/linear_probe_results.json', 'w') as f:
    json.dump(probe_results, f, indent=2)

## Cell 13 — ENSO Displacement Z-Score

Same statistic as nb15. Expected interpretation under the lat-aware redesign:
- If month-confound is fixed (F < 50) AND z is still > 5: the SSL ENSO signal is real intraseasonal ENSO modulation, not a seasonal artifact.
- If month-confound is fixed but z drops to ≈ 2: the bulk of nb15's z=13.44 was seasonal contamination, and the *true* SSL ENSO signal is comparable to the supervised baseline.
- If month-confound persists: z is uninterpretable until the seasonal contamination is resolved.

In [ ]:
labels_act = labels_bp.loc[active_mask].reset_index(drop=True)
emb_act = embeddings_2d[active_mask]

phases = range(1, 9)
disp_mag = []
for ph in phases:
    mEN = (labels_act['phase'] == ph) & (labels_act['enso_category'] == 'El Nino')
    mLN = (labels_act['phase'] == ph) & (labels_act['enso_category'] == 'La Nina')
    if mEN.sum() < 3 or mLN.sum() < 3:
        disp_mag.append(np.nan); continue
    cEN = emb_act[mEN].mean(axis=0); cLN = emb_act[mLN].mean(axis=0)
    disp_mag.append(np.linalg.norm(cEN - cLN))

rng = np.random.default_rng(42)
baseline_mag = []
for _ in range(100):
    shuf = labels_act['enso_category'].sample(frac=1, random_state=rng.integers(1e6)).values
    mtrl = []
    for ph in phases:
        mph = (labels_act['phase'] == ph).values
        mEN = mph & (shuf == 'El Nino'); mLN = mph & (shuf == 'La Nina')
        if mEN.sum() < 3 or mLN.sum() < 3: continue
        mtrl.append(np.linalg.norm(emb_act[mEN].mean(axis=0) - emb_act[mLN].mean(axis=0)))
    if mtrl: baseline_mag.append(np.mean(mtrl))

bmu = float(np.mean(baseline_mag)); bsd = float(np.std(baseline_mag))
obs_mu = float(np.nanmean(disp_mag))
z_score_ssl = float((obs_mu - bmu) / (bsd + 1e-8))

print(f'EN−LN displacement summary (MJO SSL lat16):')
print(f'  Observed mean: {obs_mu:.4f}')
print(f'  Null baseline: {bmu:.4f} ± {bsd:.4f}')
print(f'  Z-score:       {z_score_ssl:.2f}')
print(f'  (nb15 SSL z=13.44; BSISO SSL z=14.55; BSISO sup z=2.53)')

fig, ax = plt.subplots(figsize=(8, 5))
valid_p = [p for p, m in zip(phases, disp_mag) if not np.isnan(m)]
valid_m = [m for m in disp_mag if not np.isnan(m)]
ax.bar(valid_p, valid_m, color='steelblue', alpha=0.8)
ax.axhline(bmu, color='red', linestyle='--', linewidth=1.5, label=f'Null mean ({bmu:.3f})')
ax.axhline(bmu + 2*bsd, color='red', linestyle=':', linewidth=1, label='Null +2σ')
ax.axhline(obs_mu, color='steelblue', linewidth=2, label=f'Observed mean ({obs_mu:.3f})')
ax.set_xticks(range(1, 9))
ax.set_xlabel('RMM Phase'); ax.set_ylabel('||EN−LN||')
ax.set_title(f'MJO SSL (lat16) ENSO Displacement, z = {z_score_ssl:.2f}  (no ENSO labels in training)', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/enso_displacement.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 14 — MJO SSL Lat16 Summary + Auto-Decision

The decision tree explicitly compares against Session 25 targets:
- Month F < 50 = seasonal confound resolved
- Phase val > 30% = phase recovery improved over nb15 (24.7%)
- z still > 5 = SSL ENSO signal survives without seasonal contamination

In [ ]:
import time as _t

# References for context
REF = {
    'BSISO 2D SSL (nb08)':              {'phase_val': 0.30,  'phase_cv': '~30-40%',     'enso_bal': '~33%', 'z': 14.55, 'month_F': 'n/a (MJJAS only)'},
    'MJO 2D SSL — meridional avg (nb15)':{'phase_val': 0.247, 'phase_cv': '~25%',         'enso_bal': '~33%', 'z': 13.44, 'month_F': 300.84},
    'MJO 2D sup — meridional avg (nb14)':{'phase_val': 0.577, 'phase_cv': '60.3% ± 1.7%','enso_bal': 'n/a',  'z': 12.21, 'month_F': 'n/a'},
}

rows = []
for name, b in REF.items():
    rows.append([name, f"{b['phase_val']*100:.1f}%" if isinstance(b['phase_val'], float) else str(b['phase_val']),
                 b['phase_cv'], str(b['enso_bal']), f"{b['z']:.2f}", str(b['month_F'])])
rows.append(['**MJO 2D SSL — lat16 (this nb)**', f'{phase_val*100:.1f}%',
             f'{cv_phase.mean()*100:.1f}% ± {cv_phase.std()*100:.1f}%',
             f'{enso_bal*100:.1f}%', f'{z_score_ssl:.2f}', f'{f_a:.2f}'])
df_comp = pd.DataFrame(rows, columns=['Configuration', 'Phase val', 'Phase CV', 'ENSO bal', 'z-score', 'Month F'])

print('=' * 130)
print('MJO SSL LAT16 SUMMARY — Compared against nb15 (meridional avg) + BSISO references')
print('=' * 130)
print(df_comp.to_string(index=False))

# Auto-decision: Session 25 explicit targets
month_confound_resolved = f_a < 50
month_confound_improved = f_a < 200   # noticeable improvement even if not full resolution
phase_improved = phase_val > 0.30     # Session 25 target
z_survives = z_score_ssl >= 5.0       # ENSO signal survives without seasonal contamination
z_pass     = z_score_ssl >= 3.0
no_signal  = phase_val < 0.20

if not month_confound_improved:
    decision = 'CONFOUND_PERSISTS'
    decision_text = (f"**Lat-aware redesign did NOT resolve the seasonal confound** "
                     f"(month F = {f_a:.1f}; nb15 was 300.84; target < 50). "
                     f"The encoder still uses seasonal information. "
                     f"Next steps: tighter bandpass (20, 60) d, restrict negatives to same calendar month, "
                     f"or scope to DJF-only.")
elif month_confound_resolved and phase_improved and z_survives:
    decision = 'LAT16_FULL_SUCCESS'
    decision_text = (f"**Lat-aware redesign succeeded on all three Session 25 targets.** "
                     f"Month F dropped from 300.84 to {f_a:.1f} (target < 50). "
                     f"Phase val rose from 24.7% to {phase_val*100:.1f}% (target > 30%). "
                     f"z={z_score_ssl:.2f} confirms the SSL ENSO signal is real, not a seasonal artifact. "
                     f"→ Proceed to nb16b (three-way comparison) — this is the publishable MJO result.")
elif month_confound_resolved and not z_survives:
    decision = 'CONFOUND_FIXED_SIGNAL_DROPS'
    decision_text = (f"Seasonal confound resolved (F = {f_a:.1f}) but z dropped to {z_score_ssl:.2f}. "
                     f"This *confirms* that the bulk of nb15's z=13.44 was seasonal contamination, "
                     f"and the true SSL ENSO signal is closer to the supervised baseline. "
                     f"Still a publishable scientific finding — just a different framing.")
elif month_confound_improved:
    decision = 'PARTIAL_IMPROVEMENT'
    decision_text = (f"Partial improvement (month F: 300.84 → {f_a:.1f}; phase val: 24.7% → {phase_val*100:.1f}%; z = {z_score_ssl:.2f}). "
                     f"The lat-aware change helped but didn't fully resolve. "
                     f"Consider trying the SSL-only asymmetric architecture (smaller channels, see Session 25 open question) "
                     f"or tightening the bandpass.")
elif no_signal:
    decision = 'NO_SIGNAL'
    decision_text = (f"Phase recovery near random ({phase_val*100:.1f}%). Inspect training curves, bandpass output, and verify nb13b verification gate.")
else:
    decision = 'AMBIGUOUS'
    decision_text = (f"Mixed results (phase {phase_val*100:.1f}%, z={z_score_ssl:.2f}, month F={f_a:.1f}). Document and discuss before nb16b.")

print('\n' + '=' * 70)
print(f'DECISION: {decision}')
print('=' * 70)
print(decision_text)

summary = f"""# MJO SSL Temporal 2D (lat16) Summary

**Auto-generated by notebook 15b.**
**Date:** {_t.strftime('%Y-%m-%d')}
**Model:** `encoder_{RUN_TAG}_final.pth`, no L2 norm, τ={TEMPERATURE}, weight_decay={WEIGHT_DECAY}
**Architecture:** 4 lat-pool blocks (k=3×3, lat-only MaxPool) + 2 lon-pool blocks (k=1×3, lon-only MaxPool) + Linear(32, 2)
**Input shape:** (N, 3, 16, 180) — 15°S–15°N strip preserved end-to-end
**Pairs:** anchor d, positive in [d−{MAX_DELTA}, d+{MAX_DELTA}] \\ {{d}}, same year. **No RMM/ENSO labels.**
**Bandpass:** Lanczos {BP_LOW_DAYS}–{BP_HIGH_DAYS} d, half-window {BP_HALF_W} d, edge drop ±{EDGE_DROP_DAYS} d, applied per `(lat, lon)`. Result: {len(X_bp)} samples (from {len(X)}).

## Headline comparison

{df_comp.to_markdown(index=False)}

## Session 25 targets vs achieved

| Target | nb15 (meridional avg) | nb15b target | nb15b achieved |
|--------|----------------------|--------------|----------------|
| Month F (angle) | 300.84 | < 50 | **{f_a:.2f}** |
| Phase val | 24.7% | > 30% | **{phase_val*100:.1f}%** |
| z-score | 13.44 | (interpretable, > 5 if real) | **{z_score_ssl:.2f}** |

## Decision

{decision_text}

## Files produced

```
lat16/checkpoints/encoder_{RUN_TAG}_final.pth
lat16/checkpoints/training_history_{RUN_TAG}.json
lat16/data/processed/X_MJO_lat16_bp20_90.npy         ({len(X_bp)} samples, bandpassed)
lat16/data/processed/labels_aligned_mjo_lat16_bp20_90.csv
lat16/results/ssl/embeddings.npy
lat16/results/ssl/bandpass_filter_response.png
lat16/results/ssl/training_curves.png
lat16/results/ssl/embedding_2d_overview.png
lat16/results/ssl/month_clustering_check.png
lat16/results/ssl/linear_probe_results.json
lat16/results/ssl/enso_displacement.png
lat16/results/ssl/mjo_ssl_lat16_summary.md  (this file)
```

## Next

- `16b_mjo_comparison_lat16.ipynb` — three-way comparison: RMM (conventional) vs supervised-lat16 (nb14b) vs SSL-lat16 (this nb), plus ablation panels (nb15 vs nb15b for month F and phase val)
"""

with open(f'{RESULTS_DIR}/mjo_ssl_lat16_summary.md', 'w') as f:
    f.write(summary)
print(f'\nSaved: {RESULTS_DIR}/mjo_ssl_lat16_summary.md')

## Cell 15 — (Optional) Download Outputs

In [ ]:
from google.colab import files
for fname in sorted(os.listdir(RESULTS_DIR)):
    files.download(f'{RESULTS_DIR}/{fname}')

---
## Done!

**Send back:**
1. `mjo_ssl_lat16_summary.md` — the auto-decision + Session 25 targets table
2. `month_clustering_check.png` — **the key diagnostic** (did F drop from 300 → < 50?)
3. `embedding_2d_overview.png` — 4-panel scatter (the by-month panel should now be well-mixed)
4. `bandpass_filter_response.png` — sanity that the 20–90 d band is what we wanted
5. `training_curves.png` — norms stable, loss converged
6. `enso_displacement.png` — does ENSO modulation survive after seasonal confound is removed?

Then we proceed to nb16b (three-way comparison + ablation).

---
*DDCS Project | jh9141@nyu.edu*